In [0]:
from pyspark.sql.functions import col, count, when, isnotnull
from pyspark.sql.types import StringType

# List of your 5 Silver tables
silver_tables = [
    "silver_device_operations",
    "silver_environment_network",
    "silver_health_diagnostics",
    "silver_sensor_stream",
    "silver_time_events"
]

def run_silver_tests():
    print("🚀 Starting Silver Layer Quality & Logic Validation...")
    
    for table in silver_tables:
        full_name = f"iot_catalog_p2.silver.{table}"
        print(f"\n--- Testing Table: {table} ---")
        
        try:
            df = spark.table(full_name)
            
            # 1. Row Count Test
            row_count = df.count()
            assert row_count > 0, f"FAILED: {table} is empty."
            print(f"✅ Pass: {table} contains {row_count} records.")

            # 2. Corrupt String Filter Test (Checking for ?? and ***)
            # This verifies the fix for the issue you encountered earlier
            string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
            for c in string_cols:
                bad_data_count = df.filter(
                    (col(c) == "??") | (col(c) == "***") | (col(c) == "UNKNOWN")
                ).count()
                
                # We expect 0 for ?? and ***. UNKNOWN is acceptable for non-critical fields.
                strict_bad_data = df.filter((col(c) == "??") | (col(c) == "***")).count()
                assert strict_bad_data == 0, f"FAILED: Found {strict_bad_data} corrupt markers in column {c}."
            
            print(f"✅ Pass: No corrupt symbols (??, ***) found in string columns.")

            # 3. Critical Column Null Check
            # Ensures core identifiers are never null
            critical_cols = ["device_id", "sensor_identifier"] # Adjusted based on common IoT keys
            for c in [col_name for col_name in critical_cols if col_name in df.columns]:
                null_count = df.filter(col(c).isNull()).count()
                assert null_count == 0, f"FAILED: {c} contains {null_count} null values."
            print(f"✅ Pass: Critical ID columns are fully populated.")

        except Exception as e:
            print(f"❌ Error during {table} validation: {str(e)}")

# Specific Logic Tests for Specialized Tables
def run_logic_tests():
    print("\n🚀 Starting Specialized Transformation Logic Tests...")

    # Test: Anomaly Z-Score Logic (silver_sensor_stream)
    sensor_df = spark.table("iot_catalog_p2.silver.silver_sensor_stream")
    if "z_score" in sensor_df.columns:
        anomalies = sensor_df.filter("z_score > 3").count()
        print(f"✅ Pass: silver_sensor_stream anomaly logic is active ({anomalies} spikes flagged).")

    # Test: Health Indicator Resolution (silver_health_diagnostics)
    # This verifies the column resolution error we fixed earlier
    try:
        health_df = spark.table("iot_catalog_p2.silver.silver_health_diagnostics")
        assert "health_indicator" in health_df.columns or "health_status" in health_df.columns
        print("✅ Pass: Health diagnostics schema correctly resolved.")
    except:
        print("❌ Fail: Health columns not found in silver_health_diagnostics.")

# Execute All Tests
run_silver_tests()
run_logic_tests()